# LLaMA-2-7B DEWA Trajectory Profiling with Activation-Outlier Separation

Measure whether separating activation outliers lets more Group-16 partial sums use the normal
DEWA accumulation path. This notebook runs **BFP4 only**, **G = 16**, and **T = 8..12**. It does
not compute perplexity.

Every quantized `nn.Linear` is observed by one forward pre-hook. From the same raw FP16 layer
input and the same already-BFP4-quantized weight, the profiler constructs two shadow paths:

- `raw`: original activation, then BFP4 quantization.
- `clean`: activation outliers are zeroed first, then the normal activation is BFP4-quantized.

Both paths replay Group-16 partial sums in true K order and maintain independent DEWA
accumulators for every T. The clean path is diagnostic and is not propagated to downstream
layers. Outlier-lane and affected-group counts estimate the cost of a future exception path;
the exception arithmetic and final recombination are intentionally not implemented here.


In [ ]:
import gc
import json
import math
import os
import platform
import re
import time
from dataclasses import asdict, dataclass
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"


@dataclass(frozen=True)
class BFPConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 3  # BFP4 = 1 sign bit + 3 magnitude bits.
    rounding: str = "nearest"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = True

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This profiler is fixed to Group-16.")
        if self.shared_exponent_bits <= 0 or self.mantissa_bits != 3:
            raise ValueError("This profiler is fixed to BFP4 with a positive exponent width.")
        if self.rounding not in {"nearest", "trunc"}:
            raise ValueError("rounding must be 'nearest' or 'trunc'.")


@dataclass(frozen=True)
class OutlierConfig:
    method: str = "mu+sigma_k*sigma"
    sigma_k: float = 3.0
    axis: str = "tensor"

    def validate(self):
        if self.method != "mu+sigma_k*sigma" or self.axis != "tensor":
            raise ValueError("Only tensor-wise mu+sigma_k*sigma is implemented.")
        if not math.isfinite(self.sigma_k) or self.sigma_k < 0:
            raise ValueError("sigma_k must be finite and non-negative.")


@dataclass(frozen=True)
class ProfileConfig:
    thresholds: tuple = tuple(range(8, 13))
    max_profile_blocks: int = 8
    token_stride: int = 1
    out_subsample: int | None = 1024
    output_chunk: int = 256
    group_chunk: int = 8

    def validate(self):
        if self.thresholds != tuple(range(8, 13)):
            raise ValueError("This profiler is fixed to T=8..12.")
        if min(self.max_profile_blocks, self.token_stride, self.output_chunk, self.group_chunk) <= 0:
            raise ValueError("Profiling counts and chunk sizes must be positive.")
        if self.out_subsample is not None and self.out_subsample <= 0:
            raise ValueError("out_subsample must be positive or None.")


BFP = BFPConfig()
OUTLIER = OutlierConfig()
PROFILE = ProfileConfig()
BFP.validate()
OUTLIER.validate()
PROFILE.validate()

OUTPUT_PATH = Path(
    f"results/profiling/dewa-outlier/llama2-7b/bfp4-g{BFP.block_size}-t8-12.json"
)
torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

print(f"BFP config: {BFP}")
print(f"Outlier config: {OUTLIER}")
print(f"Profile config: {PROFILE}")
print(f"Output: {OUTPUT_PATH}")
if not torch.cuda.is_available():
    print("CUDA is unavailable: smoke tests can run on CPU; the full profiler cannot.")


## BFP4 fake-quantization path

Weights are BFP4-quantized once in place. The model forward uses the raw BFP4 activation path
so downstream activations remain a common baseline. The pre-hook separately quantizes the raw
and outlier-clean activation for paired trajectory profiling only.


In [ ]:
def _quantize_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    shared_exp = torch.floor(torch.log2(safe_max))

    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    shared_exp = shared_exp.clamp(exp_min, exp_max)
    shared_exp = torch.where(max_abs == 0, torch.zeros_like(shared_exp), shared_exp)

    step = torch.pow(2.0, shared_exp - (config.mantissa_bits - 1))
    mantissa = blocks / step
    mantissa = torch.round(mantissa) if config.rounding == "nearest" else torch.trunc(mantissa)
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = mantissa.clamp(-mantissa_max, mantissa_max)

    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape)
    return dequantized.to(rows.dtype)


def quantize_bfp(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.size(0) <= chunk_rows:
        return _quantize_bfp_rows(tensor, config)

    output = torch.empty_like(flat)
    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        output[start:end] = _quantize_bfp_rows(flat[start:end], config)
    return output.reshape_as(tensor)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        weight[start:end].copy_(_quantize_bfp_rows(weight[start:end], config))


class BFPLinear(nn.Module):
    def __init__(self, linear, config):
        super().__init__()
        self.linear = linear
        self.config = config

    def forward(self, x):
        x_bfp = quantize_bfp(x, self.config, self.config.activation_chunk_rows)
        return F.linear(x_bfp, self.linear.weight, self.linear.bias).to(torch.float16)


def replace_linear_layers(module, config, prefix=""):
    replaced = []
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            quantize_weight_in_place(child.weight, config)
            setattr(module, name, BFPLinear(child, config))
            replaced.append(full_name)
        else:
            replaced.extend(replace_linear_layers(child, config, full_name))
    return replaced


## Paired DEWA trajectory profiler

For every `(token, sampled output channel)` accumulator, partial sums are consumed in true K
order. Each T owns an independent accumulator because skip/replace decisions alter subsequent
`E_acc(t)`. The main comparison is the paired transition count
`raw_oow_clean_normal - raw_normal_clean_oow`. Positive values mean activation cleaning admits
more common, nonzero accumulator steps into DEWA's normal-add path.


In [ ]:
PATH_NAMES = ("raw", "clean")
STAT_NAMES = (
    "total_slots",
    "initial_or_zero_old_load",
    "zero_new",
    "skip_new",
    "replace_old",
    "normal_add",
    "total_nonzero_decisions",
    "exact_new_absorbed",
    "exact_old_absorbed",
    "oow_exact_match",
)
PAIR_STAT_NAMES = (
    "common_nonzero_decisions",
    "raw_oow_clean_normal",
    "raw_normal_clean_oow",
    "both_normal",
    "both_oow",
)
STAT_INDEX = {name: index for index, name in enumerate(STAT_NAMES)}
PAIR_STAT_INDEX = {name: index for index, name in enumerate(PAIR_STAT_NAMES)}


def evenly_spaced_indices(total, count, device=None):
    if count >= total:
        return torch.arange(total, device=device, dtype=torch.long)
    if count == 1:
        return torch.tensor([total // 2], device=device, dtype=torch.long)
    return torch.linspace(0, total - 1, steps=count, device=device).round().to(torch.long)


def _binary_exponent(value):
    tiny = torch.finfo(torch.float32).tiny
    return torch.floor(torch.log2(value.abs().clamp_min(tiny)))


def _new_state(num_thresholds, rows, cols, device):
    shape = (num_thresholds, rows, cols)
    return {
        "acc": torch.zeros(shape, dtype=torch.float32, device=device),
        "ever_decision": torch.zeros(shape, dtype=torch.bool, device=device),
        "ever_oow": torch.zeros(shape, dtype=torch.bool, device=device),
        "stats": torch.zeros(
            (num_thresholds, len(STAT_NAMES)), dtype=torch.int64, device=device
        ),
    }


def _sum_mask(mask):
    return mask.sum(dim=(1, 2), dtype=torch.int64)


def _update_state(state, new, threshold_tensor):
    old = state["acc"]
    new_view = new.unsqueeze(0)
    old_nonzero = old != 0
    new_nonzero = (new != 0).unsqueeze(0)
    load_new = ~old_nonzero & new_nonzero
    zero_new = (~new_nonzero).expand_as(old)
    both_nonzero = old_nonzero & new_nonzero

    delta = _binary_exponent(new_view) - _binary_exponent(old)
    skip_new = both_nonzero & (delta <= -threshold_tensor)
    replace_old = both_nonzero & (delta >= threshold_tensor)
    normal_add = both_nonzero & ~(skip_new | replace_old)
    oow = skip_new | replace_old

    summed = old + new_view
    exact_new_absorbed = both_nonzero & (summed == old)
    exact_old_absorbed = both_nonzero & (summed == new_view)
    oow_exact_match = (skip_new & exact_new_absorbed) | (replace_old & exact_old_absorbed)

    stats = state["stats"]
    stats[:, STAT_INDEX["total_slots"]] += new.numel()
    stats[:, STAT_INDEX["initial_or_zero_old_load"]] += _sum_mask(load_new)
    stats[:, STAT_INDEX["zero_new"]] += _sum_mask(zero_new)
    stats[:, STAT_INDEX["skip_new"]] += _sum_mask(skip_new)
    stats[:, STAT_INDEX["replace_old"]] += _sum_mask(replace_old)
    stats[:, STAT_INDEX["normal_add"]] += _sum_mask(normal_add)
    stats[:, STAT_INDEX["total_nonzero_decisions"]] += _sum_mask(both_nonzero)
    stats[:, STAT_INDEX["exact_new_absorbed"]] += _sum_mask(exact_new_absorbed)
    stats[:, STAT_INDEX["exact_old_absorbed"]] += _sum_mask(exact_old_absorbed)
    stats[:, STAT_INDEX["oow_exact_match"]] += _sum_mask(oow_exact_match)

    updated = torch.where(load_new | replace_old, new_view, old)
    state["acc"] = torch.where(normal_add, summed, updated)
    state["ever_decision"] |= both_nonzero
    state["ever_oow"] |= oow
    return {"both_nonzero": both_nonzero, "normal_add": normal_add, "oow": oow}


def _update_pair_stats(pair_stats, raw_masks, clean_masks):
    common = raw_masks["both_nonzero"] & clean_masks["both_nonzero"]
    rescued = common & raw_masks["oow"] & clean_masks["normal_add"]
    regressed = common & raw_masks["normal_add"] & clean_masks["oow"]
    both_normal = common & raw_masks["normal_add"] & clean_masks["normal_add"]
    both_oow = common & raw_masks["oow"] & clean_masks["oow"]
    masks = (common, rescued, regressed, both_normal, both_oow)
    for index, mask in enumerate(masks):
        pair_stats[:, index] += _sum_mask(mask)


def _rate(numerator, denominator):
    numerator = int(numerator)
    denominator = int(denominator)
    return {
        "numerator": numerator,
        "denominator": denominator,
        "rate": numerator / denominator if denominator else None,
    }


def _export_paths(path_stats, fit_counts, thresholds):
    exported = {}
    for path in PATH_NAMES:
        rows = []
        for index, threshold in enumerate(thresholds):
            counts = {
                name: int(path_stats[path][index, stat_index].item())
                for stat_index, name in enumerate(STAT_NAMES)
            }
            decisions = counts["total_nonzero_decisions"]
            oow = counts["skip_new"] + counts["replace_old"]
            exact_absorption = counts["exact_new_absorbed"] + counts["exact_old_absorbed"]
            rows.append({
                "threshold": int(threshold),
                "counts": counts,
                "rates": {
                    "oow_rate": _rate(oow, decisions),
                    "new_discard_rate": _rate(counts["skip_new"], decisions),
                    "old_replace_rate": _rate(counts["replace_old"], decisions),
                    "normal_add_rate": _rate(counts["normal_add"], decisions),
                    "slot_normal_add_rate": _rate(counts["normal_add"], counts["total_slots"]),
                    "all_fit_accumulator_rate": _rate(
                        fit_counts[path][index, 0], fit_counts[path][index, 1]
                    ),
                    "exact_fp32_absorption_rate": _rate(exact_absorption, decisions),
                    "oow_exact_match_rate": _rate(counts["oow_exact_match"], oow),
                },
            })
        exported[path] = rows
    return exported


def _export_pairs(pair_stats, thresholds):
    rows = []
    for index, threshold in enumerate(thresholds):
        counts = {
            name: int(pair_stats[index, stat_index].item())
            for stat_index, name in enumerate(PAIR_STAT_NAMES)
        }
        common = counts["common_nonzero_decisions"]
        rescued = counts["raw_oow_clean_normal"]
        regressed = counts["raw_normal_clean_oow"]
        rows.append({
            "threshold": int(threshold),
            "counts": counts,
            "rates": {
                "rescued_rate": _rate(rescued, common),
                "regressed_rate": _rate(regressed, common),
                "net_normal_admission_gain": _rate(rescued - regressed, common),
            },
        })
    return rows


def _validate_scope(scope, thresholds):
    expected_thresholds = list(thresholds)
    for path in PATH_NAMES:
        rows = scope["paths"][path]
        if [row["threshold"] for row in rows] != expected_thresholds:
            raise RuntimeError(f"Threshold ordering mismatch for {path}.")
        for row in rows:
            counts = row["counts"]
            if counts["total_slots"] != (
                counts["zero_new"]
                + counts["initial_or_zero_old_load"]
                + counts["total_nonzero_decisions"]
            ):
                raise RuntimeError(f"Slot accounting mismatch for {path}, T={row['threshold']}.")
            if counts["total_nonzero_decisions"] != (
                counts["skip_new"] + counts["replace_old"] + counts["normal_add"]
            ):
                raise RuntimeError(f"Decision accounting mismatch for {path}, T={row['threshold']}.")
            oow = counts["skip_new"] + counts["replace_old"]
            if counts["oow_exact_match"] > oow:
                raise RuntimeError(f"Exact-match count exceeds OOW count for {path}.")
            fit = row["rates"]["all_fit_accumulator_rate"]
            if fit["numerator"] > fit["denominator"]:
                raise RuntimeError(f"All-fit count exceeds its denominator for {path}.")

    pair_rows = scope["paired_comparison"]
    if [row["threshold"] for row in pair_rows] != expected_thresholds:
        raise RuntimeError("Paired threshold ordering mismatch.")
    for row in pair_rows:
        counts = row["counts"]
        partition = (
            counts["raw_oow_clean_normal"]
            + counts["raw_normal_clean_oow"]
            + counts["both_normal"]
            + counts["both_oow"]
        )
        if counts["common_nonzero_decisions"] != partition:
            raise RuntimeError(f"Paired accounting mismatch for T={row['threshold']}.")

    for rate in scope["exception_path_cost"].values():
        if rate["numerator"] > rate["denominator"]:
            raise RuntimeError("Exception-path count exceeds its denominator.")


def validate_export(aggregate, records, thresholds):
    _validate_scope(aggregate, thresholds)
    for record in records:
        _validate_scope(record, thresholds)


def _parse_layer_meta(name):
    match = re.search(r"model\.layers\.(\d+)\.", name)
    return name.rsplit(".", 1)[-1], int(match.group(1)) if match else -1


class DEWAOutlierProfiler:
    def __init__(self, bfp_config, outlier_config, profile_config):
        self.bfp_config = bfp_config
        self.outlier_config = outlier_config
        self.profile_config = profile_config
        self.thresholds = profile_config.thresholds
        self.layers = {}
        self.handles = []

    def _make_outlier_mask(self, flat):
        magnitude = flat.float().abs()
        mu = magnitude.mean()
        sigma = magnitude.std(unbiased=False)
        return magnitude > mu + self.outlier_config.sigma_k * sigma

    def _ensure(self, name, shape_meta):
        num_thresholds = len(self.thresholds)
        if name not in self.layers:
            self.layers[name] = {
                "shape_meta": shape_meta,
                "num_calls": 0,
                "profiled_tokens": 0,
                "path_stats": {
                    path: torch.zeros((num_thresholds, len(STAT_NAMES)), dtype=torch.int64)
                    for path in PATH_NAMES
                },
                "fit_counts": {
                    path: torch.zeros((num_thresholds, 2), dtype=torch.int64)
                    for path in PATH_NAMES
                },
                "pair_stats": torch.zeros(
                    (num_thresholds, len(PAIR_STAT_NAMES)), dtype=torch.int64
                ),
                "outlier_lanes": 0,
                "total_real_lanes": 0,
                "outlier_groups": 0,
                "total_groups": 0,
                "exception_group_output_slots": 0,
                "total_group_output_slots": 0,
            }
        record = self.layers[name]
        if record["shape_meta"] != shape_meta:
            raise RuntimeError(f"Layer shape changed while profiling {name}.")
        record["num_calls"] += 1
        return record

    @torch.no_grad()
    def _update(self, name, x, weight):
        config = self.bfp_config
        profile = self.profile_config
        width = x.shape[-1]
        flat = x.reshape(-1, width)
        if profile.token_stride > 1:
            flat = flat[::profile.token_stride]

        outlier_mask = self._make_outlier_mask(flat)
        clean = flat.masked_fill(outlier_mask, 0.0)
        quantized = {
            "raw": quantize_bfp(flat, config, config.activation_chunk_rows).float(),
            "clean": quantize_bfp(clean, config, config.activation_chunk_rows).float(),
        }

        weight_view = weight.detach()
        total_out = weight_view.shape[0]
        if profile.out_subsample is not None and total_out > profile.out_subsample:
            indices = evenly_spaced_indices(total_out, profile.out_subsample, weight_view.device)
            weight_view = weight_view.index_select(0, indices)
        weight_view = weight_view.float()

        pad = (-width) % config.block_size
        if pad:
            quantized = {path: F.pad(rows, (0, pad)) for path, rows in quantized.items()}
            weight_view = F.pad(weight_view, (0, pad))
            grouped_mask = F.pad(outlier_mask, (0, pad))
        else:
            grouped_mask = outlier_mask

        num_tokens = flat.shape[0]
        num_groups = (width + pad) // config.block_size
        profiled_out = weight_view.shape[0]
        shape_meta = (width, total_out, profiled_out, num_groups)
        record = self._ensure(name, shape_meta)
        record["profiled_tokens"] += num_tokens

        grouped_mask = grouped_mask.reshape(num_tokens, num_groups, config.block_size)
        group_has_outlier = grouped_mask.any(dim=-1)
        outlier_groups = int(group_has_outlier.sum().item())
        total_groups = group_has_outlier.numel()
        record["outlier_lanes"] += int(outlier_mask.sum().item())
        record["total_real_lanes"] += outlier_mask.numel()
        record["outlier_groups"] += outlier_groups
        record["total_groups"] += total_groups
        record["exception_group_output_slots"] += outlier_groups * profiled_out
        record["total_group_output_slots"] += total_groups * profiled_out

        grouped_x = {
            path: rows.reshape(num_tokens, num_groups, config.block_size)
                      .permute(1, 0, 2).contiguous()
            for path, rows in quantized.items()
        }
        threshold_tensor = torch.tensor(
            self.thresholds, dtype=torch.float32, device=flat.device
        ).reshape(-1, 1, 1)

        for out_start in range(0, profiled_out, profile.output_chunk):
            out_end = min(out_start + profile.output_chunk, profiled_out)
            weight_chunk = weight_view[out_start:out_end]
            grouped_weight = weight_chunk.reshape(
                out_end - out_start, num_groups, config.block_size
            ).permute(1, 2, 0).contiguous()
            states = {
                path: _new_state(
                    len(self.thresholds), num_tokens, out_end - out_start, flat.device
                )
                for path in PATH_NAMES
            }
            pair_stats = torch.zeros(
                (len(self.thresholds), len(PAIR_STAT_NAMES)),
                dtype=torch.int64,
                device=flat.device,
            )

            for group_start in range(0, num_groups, profile.group_chunk):
                group_end = min(group_start + profile.group_chunk, num_groups)
                partials = {
                    path: torch.bmm(
                        grouped_x[path][group_start:group_end],
                        grouped_weight[group_start:group_end],
                    )
                    for path in PATH_NAMES
                }
                for offset in range(group_end - group_start):
                    raw_masks = _update_state(
                        states["raw"], partials["raw"][offset], threshold_tensor
                    )
                    clean_masks = _update_state(
                        states["clean"], partials["clean"][offset], threshold_tensor
                    )
                    _update_pair_stats(pair_stats, raw_masks, clean_masks)

            for path in PATH_NAMES:
                state = states[path]
                record["path_stats"][path] += state["stats"].cpu()
                has_decision = state["ever_decision"]
                all_fit = has_decision & ~state["ever_oow"]
                record["fit_counts"][path][:, 0] += all_fit.sum(
                    dim=(1, 2), dtype=torch.int64
                ).cpu()
                record["fit_counts"][path][:, 1] += has_decision.sum(
                    dim=(1, 2), dtype=torch.int64
                ).cpu()
            record["pair_stats"] += pair_stats.cpu()

    def attach(self, model):
        for name, module in model.named_modules():
            if isinstance(module, BFPLinear):
                self.handles.append(module.register_forward_pre_hook(self._make_hook(name)))
        return self

    def _make_hook(self, name):
        def hook(module, args):
            self._update(name, args[0], module.linear.weight)
            return None
        return hook

    def remove(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

    def _export_record(self, name, record):
        input_features, total_out, profiled_out, num_groups = record["shape_meta"]
        layer_type, layer_index = _parse_layer_meta(name)
        return {
            "layer_name": name,
            "layer_type": layer_type,
            "layer_index": layer_index,
            "input_features": input_features,
            "total_output_channels": total_out,
            "profiled_output_channels": profiled_out,
            "num_groups": num_groups,
            "num_calls": record["num_calls"],
            "profiled_tokens": record["profiled_tokens"],
            "paths": _export_paths(
                record["path_stats"], record["fit_counts"], self.thresholds
            ),
            "paired_comparison": _export_pairs(record["pair_stats"], self.thresholds),
            "exception_path_cost": {
                "outlier_lane_rate": _rate(
                    record["outlier_lanes"], record["total_real_lanes"]
                ),
                "outlier_group_rate": _rate(
                    record["outlier_groups"], record["total_groups"]
                ),
                "group_output_slot_rate": _rate(
                    record["exception_group_output_slots"],
                    record["total_group_output_slots"],
                ),
            },
        }

    def export(self):
        num_thresholds = len(self.thresholds)
        aggregate_path_stats = {
            path: torch.zeros((num_thresholds, len(STAT_NAMES)), dtype=torch.int64)
            for path in PATH_NAMES
        }
        aggregate_fit_counts = {
            path: torch.zeros((num_thresholds, 2), dtype=torch.int64)
            for path in PATH_NAMES
        }
        aggregate_pair_stats = torch.zeros(
            (num_thresholds, len(PAIR_STAT_NAMES)), dtype=torch.int64
        )
        exception_totals = {
            "outlier_lanes": 0,
            "total_real_lanes": 0,
            "outlier_groups": 0,
            "total_groups": 0,
            "exception_group_output_slots": 0,
            "total_group_output_slots": 0,
        }

        records = []
        for name, record in self.layers.items():
            records.append(self._export_record(name, record))
            for path in PATH_NAMES:
                aggregate_path_stats[path] += record["path_stats"][path]
                aggregate_fit_counts[path] += record["fit_counts"][path]
            aggregate_pair_stats += record["pair_stats"]
            for key in exception_totals:
                exception_totals[key] += record[key]

        records.sort(
            key=lambda row: (row["layer_index"] < 0, row["layer_index"], row["layer_type"])
        )
        aggregate = {
            "num_layers": len(records),
            "paths": _export_paths(
                aggregate_path_stats, aggregate_fit_counts, self.thresholds
            ),
            "paired_comparison": _export_pairs(aggregate_pair_stats, self.thresholds),
            "exception_path_cost": {
                "outlier_lane_rate": _rate(
                    exception_totals["outlier_lanes"],
                    exception_totals["total_real_lanes"],
                ),
                "outlier_group_rate": _rate(
                    exception_totals["outlier_groups"],
                    exception_totals["total_groups"],
                ),
                "group_output_slot_rate": _rate(
                    exception_totals["exception_group_output_slots"],
                    exception_totals["total_group_output_slots"],
                ),
            },
        }
        return aggregate, records


## Synthetic correctness checks

These checks cover DEWA threshold boundaries, accounting identities, paired-decision
partitioning, K-padding, output chunking, and one guaranteed activation outlier.


In [ ]:
_test_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_threshold = torch.tensor([8.0], device=_test_device).reshape(-1, 1, 1)

_state = _new_state(1, 1, 1, _test_device)
_state["acc"].fill_(1.0)
_update_state(_state, torch.tensor([[2.0**-8]], device=_test_device), _threshold)
assert _state["stats"][0, STAT_INDEX["skip_new"]].item() == 1
assert _state["acc"].item() == 1.0

_state = _new_state(1, 1, 1, _test_device)
_state["acc"].fill_(2.0**-8)
_update_state(_state, torch.tensor([[1.0]], device=_test_device), _threshold)
assert _state["stats"][0, STAT_INDEX["replace_old"]].item() == 1
assert _state["acc"].item() == 1.0

_state = _new_state(1, 1, 1, _test_device)
_state["acc"].fill_(1.0)
_update_state(_state, torch.tensor([[2.0**-7]], device=_test_device), _threshold)
assert _state["stats"][0, STAT_INDEX["normal_add"]].item() == 1
assert _state["acc"].item() == 1.0 + 2.0**-7

torch.manual_seed(0)
_x = torch.ones((2, 3, 70), dtype=torch.float16, device=_test_device)
_x[0, 0, 0] = 1024.0
_linear = nn.Linear(70, 7, bias=False, device=_test_device, dtype=torch.float16)
quantize_weight_in_place(_linear.weight, BFP)
_profile_config = ProfileConfig(
    thresholds=tuple(range(8, 13)),
    max_profile_blocks=1,
    out_subsample=None,
    output_chunk=3,
    group_chunk=1,
)
_profiler = DEWAOutlierProfiler(BFP, OUTLIER, _profile_config)
_profiler._update("model.layers.0.self_attn.q_proj", _x, _linear.weight)
_aggregate, _records = _profiler.export()
validate_export(_aggregate, _records, _profile_config.thresholds)
assert len(_records) == 1
assert _aggregate["exception_path_cost"]["outlier_lane_rate"]["numerator"] == 1

_expected_slots = (
    _x.shape[0]
    * _x.shape[1]
    * _linear.out_features
    * math.ceil(70 / BFP.block_size)
)
for _path in PATH_NAMES:
    for _row in _aggregate["paths"][_path]:
        _counts = _row["counts"]
        assert _counts["total_slots"] == _expected_slots
        assert _counts["total_slots"] == (
            _counts["zero_new"]
            + _counts["initial_or_zero_old_load"]
            + _counts["total_nonzero_decisions"]
        )
        assert _counts["total_nonzero_decisions"] == (
            _counts["skip_new"] + _counts["replace_old"] + _counts["normal_add"]
        )

for _row in _aggregate["paired_comparison"]:
    _counts = _row["counts"]
    assert _counts["common_nonzero_decisions"] == (
        _counts["raw_oow_clean_normal"]
        + _counts["raw_normal_clean_oow"]
        + _counts["both_normal"]
        + _counts["both_oow"]
    )
print("Synthetic DEWA trajectory checks passed.")


## Load tokenizer and sampled WikiText-2 blocks

The profiler uses complete, non-overlapping 2048-token blocks sampled uniformly across the
test split. No labels, loss, or PPL are computed.


In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    token = getpass("HF_TOKEN: ")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
usable_tokens = input_ids.size(1) // CONTEXT_LENGTH * CONTEXT_LENGTH
total_blocks = usable_tokens // CONTEXT_LENGTH
profile_blocks = min(PROFILE.max_profile_blocks, total_blocks)
profile_block_indices = evenly_spaced_indices(total_blocks, profile_blocks).tolist()

print(
    f"WikiText-2 {SPLIT}: {input_ids.numel():,} tokens, "
    f"{total_blocks} complete blocks, sampled blocks={profile_block_indices}"
)


## Run BFP4/G16/T8-12 trajectory profiling

The model executes the raw BFP4 path only to produce common downstream activations. The hook
runs raw and clean shadow trajectories and stores aggregate plus per-layer raw counts.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("The full profiler requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False
quantized_layers = replace_linear_layers(model, BFP)
if not quantized_layers:
    raise RuntimeError("No nn.Linear layers were replaced.")
torch.cuda.empty_cache()

profiler = DEWAOutlierProfiler(BFP, OUTLIER, PROFILE).attach(model)
device = next(model.parameters()).device
torch.cuda.reset_peak_memory_stats(device)
start = time.perf_counter()
try:
    with torch.inference_mode():
        for block_index in tqdm(profile_block_indices, desc="Profiling BFP4 DEWA trajectories"):
            begin = block_index * CONTEXT_LENGTH
            batch = input_ids[:, begin:begin + CONTEXT_LENGTH].to(device)
            model(batch, use_cache=False)
finally:
    profiler.remove()
torch.cuda.synchronize(device)
elapsed = time.perf_counter() - start
peak_memory_gib = torch.cuda.max_memory_allocated(device) / (1024 ** 3)

aggregate, records = profiler.export()
validate_export(aggregate, records, PROFILE.thresholds)
if len(records) != len(quantized_layers):
    raise RuntimeError("Not every quantized Linear layer was profiled.")
if any(record["num_calls"] != profile_blocks for record in records):
    raise RuntimeError("Unexpected per-layer profiling call count.")

payload = {
    "metadata": {
        "model": MODEL_ID,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "split": SPLIT,
        "analysis": "paired raw/activation-outlier-clean DEWA accumulator trajectories",
        "purpose": "measure whether activation-outlier separation admits more steps to DEWA normal_add",
        "ppl_evaluated": False,
        "format": "BFP4 (1S3M + shared E5)",
        "bfp_config": asdict(BFP),
        "outlier_config": asdict(OUTLIER),
        "profile_config": asdict(PROFILE),
        "dewa_rule": "delta=E_new-E_acc(t); skip_new if delta<=-T; replace_old if delta>=T; otherwise normal_add",
        "raw_path": f"raw FP16 layer input -> activation BFP4 -> ordered Group-{BFP.block_size} DEWA trajectory",
        "clean_path": f"raw FP16 outliers zeroed -> normal activation BFP4 -> independent ordered Group-{BFP.block_size} DEWA trajectory",
        "exception_path": "cost accounting only; outlier arithmetic and recombination are not modeled",
        "clean_path_propagated_downstream": False,
        "same_forward_call_comparison": True,
        "weight_path": "same already-BFP4-quantized weight for raw and clean paths",
        "group_partial_format": "FP32 torch.bmm with TF32 disabled",
        "accumulator_format": "FP32 functional DEWA trajectory",
        "evaluation_protocol": EVALUATION_PROTOCOL,
        "context_length": CONTEXT_LENGTH,
        "stride": STRIDE,
        "drop_remainder": DROP_REMAINDER,
        "source_input_tokens": int(input_ids.numel()),
        "complete_blocks": total_blocks,
        "profile_blocks": profile_blocks,
        "profile_block_indices": profile_block_indices,
        "profiled_input_tokens": profile_blocks * CONTEXT_LENGTH,
        "output_channel_sampling": "all_if_within_cap_else_evenly_spaced_inclusive",
        "quantized_linear_layers": len(quantized_layers),
        "elapsed_seconds": elapsed,
        "peak_gpu_memory_gib": peak_memory_gib,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
    },
    "aggregate": aggregate,
    "layers": records,
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"Layers={len(records)}, elapsed={elapsed:.1f}s, peak GPU={peak_memory_gib:.2f} GiB")


## Compact decision summary

A useful result requires lower clean OOW, higher clean normal-add admission, and a positive
paired net admission gain. The exception-path rates show the routing cost that must eventually
be included in an FP-Acc power model.


In [ ]:
raw_by_t = {row["threshold"]: row for row in aggregate["paths"]["raw"]}
clean_by_t = {row["threshold"]: row for row in aggregate["paths"]["clean"]}
pair_by_t = {row["threshold"]: row for row in aggregate["paired_comparison"]}

print("T  Raw OOW   Clean OOW   Raw normal  Clean normal  Paired net gain")
for threshold in PROFILE.thresholds:
    raw = raw_by_t[threshold]["rates"]
    clean = clean_by_t[threshold]["rates"]
    paired = pair_by_t[threshold]["rates"]
    print(
        f"{threshold:>2} "
        f"{raw['oow_rate']['rate'] or 0:>9.4%} "
        f"{clean['oow_rate']['rate'] or 0:>11.4%} "
        f"{raw['normal_add_rate']['rate'] or 0:>11.4%} "
        f"{clean['normal_add_rate']['rate'] or 0:>13.4%} "
        f"{paired['net_normal_admission_gain']['rate'] or 0:>15.4%}"
    )

exception = aggregate["exception_path_cost"]
print(
    "Outlier lanes: "
    f"{exception['outlier_lane_rate']['numerator']:,}/"
    f"{exception['outlier_lane_rate']['denominator']:,} "
    f"({exception['outlier_lane_rate']['rate']:.4%})"
)
print(
    f"Affected G{BFP.block_size} groups: "
    f"{exception['outlier_group_rate']['numerator']:,}/"
    f"{exception['outlier_group_rate']['denominator']:,} "
    f"({exception['outlier_group_rate']['rate']:.4%})"
)


In [ ]:
if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f"Profiler output does not exist: {OUTPUT_PATH}")

try:
    from google.colab import files
except ImportError:
    print(f"Not running in Colab. JSON remains at: {OUTPUT_PATH.resolve()}")
else:
    files.download(str(OUTPUT_PATH))
